In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_parquet("../data/processed/transactions_clean.parquet")
df["line_total"] = df["Quantity"] * df["Price"]
print(df.shape, df["Customer_ID"].nunique(), "customers")

(790721, 10) 5852 customers


In [3]:
# Analysis date = day after the last transaction in the data
ANALYSIS_DATE = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Analysis date:", ANALYSIS_DATE)

Analysis date: 2011-12-10 12:50:00


In [ ]:
# --- RFM ---
per_invoice = df.groupby(["Customer_ID", "Invoice"]).agg(
    invoice_date=("InvoiceDate", "max"),
    invoice_total=("line_total", "sum"),
    n_items=("StockCode", "size"),
).reset_index()

rfm = per_invoice.groupby("Customer_ID").agg(
    recency=("invoice_date", lambda s: (ANALYSIS_DATE - s.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("invoice_total", "sum"),
)
rfm.describe()

In [ ]:
# --- Extra features ---
extras = df.groupby("Customer_ID").agg(
    category_diversity=("StockCode", "nunique"),
)
extras["avg_basket_size"] = per_invoice.groupby("Customer_ID")["n_items"].mean()

is_intl = df.groupby("Customer_ID")["Country"].agg(lambda s: (s != "United Kingdom").any())
extras["is_international"] = is_intl.astype(int)

extras.describe()

In [ ]:
features = rfm.join(extras)
features.head()

In [ ]:
# Confirm the skew before transforming
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
features["monetary"].hist(bins=50, ax=axes[0])
axes[0].set_title("Monetary (raw)")
features["frequency"].hist(bins=50, ax=axes[1])
axes[1].set_title("Frequency (raw)")
plt.tight_layout()
plt.show()

In [ ]:

features_log = features.copy()
features_log["monetary"] = np.log1p(features_log["monetary"])
features_log["frequency"] = np.log1p(features_log["frequency"])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
features_log["monetary"].hist(bins=50, ax=axes[0])
axes[0].set_title("Monetary (log1p)")
features_log["frequency"].hist(bins=50, ax=axes[1])
axes[1].set_title("Frequency (log1p)")
plt.tight_layout()
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(features_log),
    columns=features_log.columns,
    index=features_log.index,
)
X_scaled.describe()

In [ ]:
OUT_DIR = Path("../data/processed")
features.to_parquet(OUT_DIR / "rfm_features.parquet")       # raw, for persona writeup later
X_scaled.to_parquet(OUT_DIR / "rfm_features_scaled.parquet") # for clustering
print("Saved.")